In [0]:
# -------------------------------
# Imports & Silver table loading
# -------------------------------

import pyspark.sql.functions as fs

df_silver = spark.read.table("chess_games.silver.games")

df_silver = df_silver.withColumn('year_month', fs.date_format(fs.col('real_timestamp'), 'yyyy-MM'))

In [0]:
# -------
# Result
# -------

df_result = df_silver.groupBy("result", 'color').count()
df_result.write.format("delta").mode("overwrite").saveAsTable("chess_games.gold.result")

In [0]:
# ------------
# Time result
# ------------

df_timed_results = df_silver.groupBy('year_month', 'result').count().orderBy('year_month', 'count', ascending=True)
df_timed_results.write.format("delta").mode("overwrite").saveAsTable("chess_games.gold.timed_results")


In [0]:
# ---------
# Accuracy
# ---------

df_accuracy = df_silver \
    .filter(fs.col('my_accuracy').isNotNull()) \
    .groupBy('year_month') \
    .avg('my_accuracy', 'rating') \
    .withColumn('Précision_moyenne', fs.round('avg(my_accuracy)', 2)) \
    .withColumn('Classement_moyen', fs.round('avg(rating)', 0)) \
    .orderBy('year_month', ascending=True) \
    .select('year_month', 'Précision_moyenne', 'Classement_moyen',)

df_accuracy.write.format("delta").mode("overwrite").saveAsTable("chess_games.gold.accuracy")

In [0]:
# -------------
# Elo interval
# -------------

df_silver = df_silver \
    .withColumn('elo_interval', fs.floor(fs.round("opponent_rating", 0) / 50) * 50)

df_elo = df_silver \
    .groupBy('elo_interval', 'result') \
    .count() \
    .orderBy('elo_interval', ascending=True)

df_elo.write.format("delta").mode("overwrite").saveAsTable("chess_games.gold.opponent_elo")

In [0]:
# ---------------
# Opening groups 
# ---------------

df_opening_groups = df_silver.groupBy("opening_group", "result") \
    .count() \
    .orderBy("opening_group", ascending=True)

from pyspark.sql.window import Window
# Définir une fenêtre pour calculer la somme des count par opening_group
window = Window.partitionBy("opening_group")

# Ajouter une colonne avec la somme des count par opening_group
df_opening_groups = df_opening_groups \
    .withColumn("total_count", fs.sum("count").over(window)) \
    .withColumn("ratio", fs.round(100 * fs.col("count") / fs.col("total_count"), 2))

# Ordre result
order_mapping = fs.when(fs.col("result") == "win", 1) \
                  .when(fs.col("result") == "loss", 2) \
                  .when(fs.col("result") == "draw", 3)

df_opening_groups = df_opening_groups \
    .filter(fs.col("total_count") > 10) \
    .orderBy(
        fs.desc("total_count"), 
        fs.desc("opening_group"), 
        order_mapping.asc())

df_opening_groups.write.format("delta").mode("overwrite").saveAsTable("chess_games.gold.opening_groups")